In [1]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/JDK_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Mounted at /content/drive


In [2]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)
# =====================================================
# 1. Imports
# =====================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import classification_report

# =====================================================
# 2. Create Labels (NDVI drop & BSI rise)
# =====================================================
# X_raw shape: (samples, time_steps, features)
# locations shape: (samples, 2) -> latitude, longitude

ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # NDVI 2021-2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # BSI 2024-2021

# Use 85th percentile thresholds (adjust if too small)
ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# =====================================================
# 3. Train-Test Split (Safe for small datasets)
# =====================================================
min_class_count = np.min([np.sum(y==0), np.sum(y==1)])
if min_class_count < 2:
    print("WARNING: Tiny dataset, skipping stratified split.")
    X_train, X_test, y_train, y_test, loc_train, loc_test = (
        X_raw, X_raw, y, y, locations, locations
    )
else:
    X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
        X_raw, y, locations,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# =====================================================
# 4. Scaling (No data leakage)
# =====================================================
scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# =====================================================
# 5. LSTM Model
# =====================================================
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Recall', 'Precision']
)

model.summary()

# =====================================================
# 6. Train Model (Use class weighting if enough samples)
# =====================================================
if min_class_count < 2:
    print("Skipping class weighting due to tiny dataset.")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=1,  # small batch for tiny data
        verbose=1
    )
else:
    class_weight = {0: 1, 1: 12}
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=128,
        class_weight=class_weight,
        verbose=1
    )

# =====================================================
# 7. Evaluation
# =====================================================
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)  # recall-friendly threshold

print(classification_report(y_test, y_pred))

# =====================================================
# 8. Save Predictions (with locations)
# =====================================================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/JDK_Deforestation_Predictions.csv',
    index=False
)

print("✅ Saved: JDK_Deforestation_Predictions.csv")

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000000a6b_0 -0.204722       1  0.482358  0.224623  0.632457   
1  00000000000000000a6b_0 -0.140087       1  0.369677  0.182838  0.519517   
2  00000000000000000a6b_0 -0.094227       1  0.304131  0.149176  0.486496   
3  00000000000000000a6c_0 -0.166187       1  0.471480  0.212036  0.568663   
4  00000000000000000a6c_0 -0.066844       1  0.271460  0.114971  0.409058   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  87.152303   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  87.152303   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  87.152303   
3  2021  {"geodesic":false,"type":"Point","coordinates"...  87.230457   
4  2022  {"geodesic":false,"type":"Point","coordinates"...  87.230457   

    latitude  
0  23.812092  
1  23.812092  
2  23.812092  
3  23.823770  
4  23.823770  
[2021 20

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - Precision: 0.1241 - Recall: 0.9919 - accuracy: 0.1296 - loss: 1.5631 - val_Precision: 0.1240 - val_Recall: 1.0000 - val_accuracy: 0.1240 - val_loss: 0.8679
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - Precision: 0.1241 - Recall: 1.0000 - accuracy: 0.1243 - loss: 1.5258 - val_Precision: 0.1241 - val_Recall: 1.0000 - val_accuracy: 0.1245 - val_loss: 0.8648
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - Precision: 0.1830 - Recall: 0.9406 - accuracy: 0.4712 - loss: 1.2137 - val_Precision: 0.3299 - val_Recall: 0.8911 - val_accuracy: 0.7620 - val_loss: 0.4875
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - Precision: 0.3442 - Recall: 0.9154 - accuracy: 0.7730 - loss: 0.7987 - val_Precision: 0.4988 - val_Recall: 0.8669 - val_accuracy: 0.8755 - val_loss: 0.2895
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - Precision: 0.4050 - Recall: 0.9376 - accuracy: 0.8213 - loss: 0.6697 - val_Precision: 0.3704 - val_Recall: 0.96

In [3]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/JDK_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/JDK_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/JDK_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())
import pandas as pd
import folium
from folium.plugins import MarkerCluster

import pandas as pd
import folium
from folium.plugins import MarkerCluster

# ===========================
# Jharkhand map
# Approx latitude: 22–25, longitude: 83–88
# ===========================
m = folium.Map(location=[23.6, 85.3], zoom_start=7)

df = pd.read_csv('/content/drive/MyDrive/JDK_Deforestation_Causes_Adaptive.csv')

# Print cause counts
print(df['cause'].value_counts())

# Define colors for causes
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

# Cluster points for performance
marker_cluster = MarkerCluster().add_to(m)

# Add points to map
for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')  # default blue if cause not in dict

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)

# Optional: Save map
m.save('/content/drive/MyDrive/jdk_adaptive.html')
print("✅ WB map saved successfully!")

NDVI: {'mean': np.float64(-0.11518547010643333), 'std': 0.11417614627236725, 'q1': np.float64(-0.18668495), 'q3': np.float64(-0.022361392499999994), 'lower_2std': np.float64(-0.3435377626511678), 'upper_2std': np.float64(0.11316682243830116)}
NBR : {'mean': np.float64(-0.11522271309146144), 'std': 0.1521969543962955, 'q1': np.float64(-0.21251834500000003), 'q3': np.float64(0.023100624999999993), 'lower_2std': np.float64(-0.4196166218840524), 'upper_2std': np.float64(0.18917119570112956)}
BSI : {'mean': np.float64(0.059247912235786404), 'std': 0.0940475534594372, 'q1': np.float64(-0.021001912440103375), 'q3': np.float64(0.12210333193144339), 'lower_2std': np.float64(-0.128847194683088), 'upper_2std': np.float64(0.2473430191546608)}
NDMI: {'mean': np.float64(-0.05683133382760852), 'std': 0.09308800697175308, 'q1': np.float64(-0.11303629300000001), 'q3': np.float64(0.015033465000000006), 'lower_2std': np.float64(-0.24300734777111468), 'upper_2std': np.float64(0.12934468011589764)}

Defore

In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv(
    '/content/drive/MyDrive/JDK_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv(
    '/content/drive/MyDrive/JDK_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())

# Ensure latitude/longitude are float to avoid merge issues
df_deforest['latitude'] = df_deforest['latitude'].astype(float)
df_deforest['longitude'] = df_deforest['longitude'].astype(float)
df_cause['latitude'] = df_cause['latitude'].astype(float)
df_cause['longitude'] = df_cause['longitude'].astype(float)

# ==============================
# LOAD JHARKHAND DISTRICTS
# ==============================
jdk = gpd.read_file('/content/drive/MyDrive/JDK.geojson')

# CRS SAFETY
if jdk.crs is None:
    jdk.set_crs("EPSG:4326", inplace=True)

jdk = jdk.to_crs("EPSG:4326")
jdk["state"] = "Jharkhand"

print(jdk.head())

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [Point(xy) for xy in zip(df_deforest['longitude'], df_deforest['latitude'])]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 2000
    latitude  longitude  deforestation
0  24.493914  87.373289              0
1  24.633153  87.456832              1
2  24.488524  87.375085              0
3  24.458879  87.213389              0
4  24.608000  87.413713              0
Total deforestation samples: 415
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  24.633153  87.456832              1    -0.101248   -0.118147    0.034247   
1  24.493914  87.174761              1    -0.077159   -0.080576    0.047396   
2  24.462473  87.405628              1    -0.155297   -0.087903    0.034948   
3  24.485829  87.426289              1    -0.145548   -0.105688    0.016609   
4  24.672678  87.455934              1    -0.165625   -0.106350    0.041007   

   NDMI_change        cause  
0    -0.019603  Urban/Other  
1    -0.037602  Urban/Other  
2    -0.009052  Agriculture  
3     0.011887  Agriculture  
4    -0.033824  Agriculture  
  REMARKS_2 Country State_Name State_Code Dist_Name Dist_C

In [9]:
import geopandas as gpd

# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================
gdf_joined = gpd.sjoin(
    gdf_points,
    jdk,                # Jharkhand GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())
print(gdf_joined['Dist_Name'].value_counts())

# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================
if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# Cleanup duplicate columns
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)

# ==============================
# DISTRICT SUMMARY — Jharkhand
# ==============================
district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/JDK_District_Wise_Deforestation.csv',
    index=False
)

print("Saved Jharkhand district summary")

# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================
cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/JDK_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved Jharkhand cause summary")

    latitude  longitude  deforestation        cause  \
0  24.633153  87.456832              1  Urban/Other   
1  24.493914  87.174761              1  Urban/Other   
2  24.462473  87.405628              1  Agriculture   
3  24.485829  87.426289              1  Agriculture   
4  24.672678  87.455934              1  Agriculture   

                    geometry  index_right REMARKS_2 Country State_Name  \
0  POINT (87.45683 24.63315)           14      None   India  Jharkhand   
1  POINT (87.17476 24.49391)            4      None   India  Jharkhand   
2  POINT (87.40563 24.46247)            4      None   India  Jharkhand   
3  POINT (87.42629 24.48583)            4      None   India  Jharkhand   
4  POINT (87.45593 24.67268)            7      None   India  Jharkhand   

  State_Code Dist_Name Dist_Code      state  
0         20     Pakur       353  Jharkhand  
1         20     Dumka       362  Jharkhand  
2         20     Dumka       362  Jharkhand  
3         20     Dumka       362  Jharkh

In [10]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Jharkhand District Boundaries
# =====================================================
jdk = gpd.read_file('/content/drive/MyDrive/JDK.geojson')

# CRS safety
if jdk.crs is None:
    jdk.set_crs(epsg=4326, inplace=True)

jdk = jdk.to_crs(epsg=4326)
jdk["state"] = "Jharkhand"
gdf_districts = jdk.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/JDK_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# Merge
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 7: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 8: Create Folium Map (center Jharkhand)
# =====================================================
m = folium.Map(location=[23.6, 85.3], zoom_start=7)

# =====================================================
# STEP 9: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Jharkhand)"
).add_to(m)

# =====================================================
# STEP 10: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 11: Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Jharkhand Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[23.6, 85.3],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 12: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/jdk_forest.html")

print("✅ Jharkhand map saved successfully with afforestation recommendation!")

/tmp/ipykernel_176/4261699892.py:77: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Jharkhand map saved successfully with afforestation recommendation!
